# 9.2 故障排查与调试方法 (Troubleshooting & Debugging)

## 概述

本 Notebook 系统性地介绍 LLM 端侧部署中常见的故障类型及其排查方法，涵盖以下主题：

1. **内存问题** — OOM、内存泄漏、KV Cache 膨胀
2. **精度异常** — 量化误差、异常通道、混合精度策略
3. **性能瓶颈** — 算子耗时、MAC 利用率、吞吐分析
4. **框架错误** — ONNX 导出、算子分解、CPU 回退
5. **端到端诊断工作流** — 综合诊断报告与决策树
6. **调试工具参考** — PyTorch Profiler、ONNX Runtime Profiling 等

所有代码均可在 CPU 上运行，使用小型虚拟模型。

---

## 1. 导入依赖与诊断工具

本节导入所有必需的库，并构建三个核心诊断工具：
- **MemoryTracker** — 跟踪 GPU/CPU 内存使用
- **LayerProfiler** — 记录每层的执行时间
- **PrecisionChecker** — 比较 FP 与量化输出差异

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import time
import gc
import json
import warnings
from collections import OrderedDict, defaultdict
from typing import Dict, List, Tuple, Optional, Any
from dataclasses import dataclass, field
from contextlib import contextmanager

warnings.filterwarnings('ignore')

print(f"PyTorch 版本: {torch.__version__}")
print(f"设备: {'CUDA' if torch.cuda.is_available() else 'CPU'}")

In [ ]:
# ============================================================
# 诊断工具 1: MemoryTracker — 内存使用跟踪器
# ============================================================

@dataclass
class MemorySnapshot:
    """内存快照数据类"""
    timestamp: float
    label: str
    allocated_gb: float
    reserved_gb: float
    cpu_ram_gb: float = 0.0


class MemoryTracker:
    """
    内存使用跟踪器，支持记录和对比内存快照。
    同时兼容 CPU 和 GPU 环境。
    """

    def __init__(self):
        self.snapshots: List[MemorySnapshot] = []
        self._start_mem = self._get_current_memory()

    @staticmethod
    def _get_current_memory() -> Tuple[float, float]:
        """获取当前内存使用 (allocated_gb, reserved_gb)"""
        if torch.cuda.is_available():
            allocated = torch.cuda.memory_allocated() / (1024 ** 3)
            reserved = torch.cuda.memory_reserved() / (1024 ** 3)
        else:
            allocated = 0.0
            reserved = 0.0
        return allocated, reserved

    def snapshot(self, label: str):
        """记录一个内存快照"""
        allocated, reserved = self._get_current_memory()
        self.snapshots.append(MemorySnapshot(
            timestamp=time.time(),
            label=label,
            allocated_gb=allocated,
            reserved_gb=reserved
        ))

    def reset_baseline(self):
        """重置基线"""
        self._start_mem = self._get_current_memory()
        self.snapshots.clear()

    def report(self) -> str:
        """生成内存使用报告"""
        if not self.snapshots:
            return "暂无内存快照记录。"

        lines = ["=" * 60, "内存使用报告", "=" * 60]
        prev_allocated = self._start_mem[0]
        for snap in self.snapshots:
            delta = snap.allocated_gb - prev_allocated
            sign = "+" if delta >= 0 else ""
            lines.append(
                f"  [{snap.label}] "
                f"已分配: {snap.allocated_gb:.4f} GB "
                f"(Δ{sign}{delta:.4f} GB)"
            )
            prev_allocated = snap.allocated_gb
        lines.append("=" * 60)
        return "\n".join(lines)


# 测试 MemoryTracker
mem_tracker = MemoryTracker()
mem_tracker.snapshot("初始状态")
test_tensor = torch.randn(1000, 1000)
mem_tracker.snapshot("创建 1000x1000 张量后")
del test_tensor
mem_tracker.snapshot("释放张量后")
print(mem_tracker.report())

In [ ]:
# ============================================================
# 诊断工具 2: LayerProfiler — 逐层性能分析器
# ============================================================

@dataclass
class LayerProfile:
    """层性能数据"""
    name: str
    time_ms: float
    flops: int = 0
    params: int = 0
    input_shape: Tuple = ()


class LayerProfiler:
    """
    逐层性能分析器。通过注册 hook 测量每个子模块的执行时间。
    """

    def __init__(self):
        self.profiles: Dict[str, LayerProfile] = {}
        self._handles = []
        self._start_events: Dict[str, float] = {}

    def _forward_pre_hook(self, name: str):
        def hook(module, input):
            self._start_events[name] = time.time()
            if name not in self.profiles:
                params = sum(p.numel() for p in module.parameters())
                input_shape = tuple(input[0].shape) if input else ()
                self.profiles[name] = LayerProfile(
                    name=name, time_ms=0.0, params=params,
                    input_shape=input_shape
                )
        return hook

    def _forward_hook(self, name: str):
        def hook(module, input, output):
            elapsed = (time.time() - self._start_events.get(name, time.time())) * 1000
            if name in self.profiles:
                self.profiles[name].time_ms += elapsed
        return hook

    def attach(self, model: nn.Module):
        """将 profiler hooks 注册到模型的所有子模块"""
        for name, module in model.named_modules():
            if name == "":
                continue
            self._handles.append(
                module.register_forward_pre_hook(self._forward_pre_hook(name))
            )
            self._handles.append(
                module.register_forward_hook(self._forward_hook(name))
            )

    def detach(self):
        """移除所有 hooks"""
        for handle in self._handles:
            handle.remove()
        self._handles.clear()

    def report(self, top_k: int = 5) -> str:
        """生成耗时 Top-K 报告"""
        sorted_items = sorted(
            self.profiles.items(),
            key=lambda x: x[1].time_ms, reverse=True
        )
        lines = ["=" * 70, f"逐层耗时报告 (Top-{top_k})", "=" * 70]
        lines.append(f"{'层名称':<35} {'耗时(ms)':<12} {'参数数量':<12} {'输入形状'}")
        lines.append("-" * 70)
        for i, (name, prof) in enumerate(sorted_items[:top_k]):
            lines.append(
                f"{name:<35} {prof.time_ms:<12.4f} {prof.params:<12} {str(prof.input_shape)}"
            )
        total_time = sum(p.time_ms for p in self.profiles.values())
        lines.append("-" * 70)
        lines.append(f"总耗时: {total_time:.4f} ms | 层数: {len(self.profiles)}")
        lines.append("=" * 70)
        return "\n".join(lines)

    def get_top_bottlenecks(self, k: int = 5) -> List[LayerProfile]:
        """获取最耗时的 k 个层"""
        sorted_items = sorted(
            self.profiles.values(),
            key=lambda x: x.time_ms, reverse=True
        )
        return sorted_items[:k]


print("LayerProfiler 工具已就绪。")

In [ ]:
# ============================================================
# 诊断工具 3: PrecisionChecker — 精度检查器
# ============================================================

@dataclass
class PrecisionReport:
    """精度报告数据类"""
    layer_name: str
    cosine_sim: float
    mse: float
    max_abs_error: float
    outlier_ratio: float
    recommendation: str


class PrecisionChecker:
    """
    精度检查器：比较 FP32 参考输出与量化/压缩后的输出之间的差异。
    """

    def __init__(self, fp_model: nn.Module, quantized_model: nn.Module):
        self.fp_model = fp_model
        self.quantized_model = quantized_model
        self.reports: Dict[str, PrecisionReport] = {}
        self._fp_outputs: Dict[str, torch.Tensor] = {}
        self._q_outputs: Dict[str, torch.Tensor] = {}

    @staticmethod
    def cosine_similarity(a: torch.Tensor, b: torch.Tensor) -> float:
        """计算两个张量之间的余弦相似度"""
        a_flat = a.float().reshape(-1)
        b_flat = b.float().reshape(-1)
        cos_sim = F.cosine_similarity(
            a_flat.unsqueeze(0), b_flat.unsqueeze(0)
        )
        return cos_sim.item()

    @staticmethod
    def outlier_channel_ratio(
        tensor: torch.Tensor, threshold: float = 5.0
    ) -> float:
        """
        检测异常通道比例。
        异常通道 = 该通道的绝对值最大值超过全局均值 N 个标准差的通道。
        """
        if tensor.dim() < 2:
            return 0.0
        channel_max = tensor.float().abs().max(dim=-1).values
        if tensor.dim() > 2:
            channel_max = channel_max.max(dim=-1).values
        global_mean = channel_max.mean()
        global_std = channel_max.std() + 1e-8
        outliers = (channel_max > global_mean + threshold * global_std).float()
        return outliers.mean().item()

    def compare_layer(self, name: str, fp_out: torch.Tensor,
                      q_out: torch.Tensor) -> PrecisionReport:
        """比较单层的 FP 与量化输出"""
        cos_sim = self.cosine_similarity(fp_out, q_out)
        mse = F.mse_loss(fp_out.float(), q_out.float()).item()
        max_err = (fp_out.float() - q_out.float()).abs().max().item()
        outlier_r = self.outlier_channel_ratio(fp_out)

        if cos_sim < 0.99:
            recommendation = "建议保留 FP16"
        elif cos_sim < 0.999:
            recommendation = "可尝试 INT8，需验证"
        else:
            recommendation = "可安全量化至 INT8"

        report = PrecisionReport(
            layer_name=name, cosine_sim=cos_sim, mse=mse,
            max_abs_error=max_err, outlier_ratio=outlier_r,
            recommendation=recommendation
        )
        self.reports[name] = report
        return report

    def report_all(self) -> str:
        """生成所有层的精度报告"""
        lines = ["=" * 75, "精度比较报告", "=" * 75]
        lines.append(
            f"{'层名称':<25} {'余弦相似度':<12} {'MSE':<12} "
            f"{'最大误差':<12} {'异常比例':<10} {'建议'}"
        )
        lines.append("-" * 75)
        for name, r in self.reports.items():
            lines.append(
                f"{name:<25} {r.cosine_sim:<12.6f} {r.mse:<12.6f} "
                f"{r.max_abs_error:<12.6f} {r.outlier_ratio:<10.4f} "
                f"{r.recommendation}"
            )
        lines.append("=" * 75)
        return "\n".join(lines)


print("PrecisionChecker 工具已就绪。")

---

## 2. 内存问题排查 (Memory Issues)

端侧部署最常见的问题就是 OOM（Out of Memory）。本节演示如何：
- 模拟 OOM 场景
- 跟踪 KV Cache 内存占用
- 检测内存泄漏
- 计算不同精度下的模型大小

In [ ]:
# ============================================================
# 2.1 模拟 OOM 场景并监控内存
# ============================================================

def simulate_oom_risk(model_size_gb: float, kv_cache_gb: float,
                       available_memory_gb: float = 8.0):
    """
    模拟 OOM 风险评估。
    端侧设备内存通常有限（4GB-16GB），模型 + KV Cache 很容易超限。
    """
    total = model_size_gb + kv_cache_gb
    overhead = total * 0.2
    total_with_overhead = total + overhead

    print(f"{'='*50}")
    print(f"OOM 风险评估")
    print(f"{'='*50}")
    print(f"  可用内存:        {available_memory_gb:.2f} GB")
    print(f"  模型大小:        {model_size_gb:.2f} GB")
    print(f"  KV Cache:        {kv_cache_gb:.2f} GB")
    print(f"  运行时开销(20%):  {overhead:.2f} GB")
    print(f"  总需求:          {total_with_overhead:.2f} GB")
    print(f"  剩余:            {available_memory_gb - total_with_overhead:.2f} GB")
    print(f"  {'='*50}")

    if total_with_overhead > available_memory_gb * 0.8:
        print(f"  ⚠️  警告: 内存使用率超过 80%!")
        print(f"  建议: 使用更小的模型或减少 KV Cache 大小")
    elif total_with_overhead > available_memory_gb:
        print(f"  ❌ 错误: 超出可用内存，必然 OOM!")
        print(f"  建议: 切换至更小的模型或使用 INT4 量化")
    else:
        print(f"  ✅ 内存充足，预计不会 OOM")

    return total_with_overhead <= available_memory_gb


# 测试：7B 模型 + KV Cache 在 8GB 设备上
# FP16 7B 模型 ≈ 14GB，必定 OOM
print("场景1: 7B FP16 模型 + KV Cache 在 8GB 设备上")
simulate_oom_risk(model_size_gb=14.0, kv_cache_gb=0.5, available_memory_gb=8.0)

print("\n场景2: 1B INT4 模型 + KV Cache 在 8GB 设备上")
simulate_oom_risk(model_size_gb=0.5, kv_cache_gb=0.2, available_memory_gb=8.0)

print("\n场景3: 3B INT8 模型 + 大 KV Cache 在 4GB 设备上")
simulate_oom_risk(model_size_gb=3.0, kv_cache_gb=1.5, available_memory_gb=4.0)

In [ ]:
# ============================================================
# 2.2 KV Cache 内存跟踪
# ============================================================

def estimate_kv_cache_memory(
    num_layers: int,
    num_heads: int,
    head_dim: int,
    max_seq_len: int,
    batch_size: int = 1,
    dtype_size: int = 2
) -> Dict[str, float]:
    """
    估算 KV Cache 内存占用。

    参数:
        num_layers: Transformer 层数
        num_heads: 注意力头数（KV 头数，如果使用 GQA 则需调整）
        head_dim: 每个头的维度
        max_seq_len: 最大序列长度
        batch_size: 批次大小
        dtype_size: 数据类型字节数 (FP16=2, FP32=4, INT8=1)

    公式: KV Cache = 2 * batch * num_layers * num_heads * head_dim * max_seq_len * dtype_size
    """
    per_token_kv = 2 * num_layers * num_heads * head_dim * dtype_size
    total_kv = per_token_kv * max_seq_len * batch_size

    result = {
        "每 token KV Cache (bytes)": per_token_kv,
        "最大序列 KV Cache (MB)": total_kv / (1024 ** 2),
        "最大序列 KV Cache (GB)": total_kv / (1024 ** 3),
        "每层 KV Cache (MB)": (per_token_kv * max_seq_len) / (1024 ** 2) / num_layers,
    }
    return result


# 示例：典型 7B LLaMA 风格模型
print("典型 7B 模型 KV Cache 估算:")
print("  配置: 32层, 32个头, head_dim=128, seq_len=4096")
kv_info = estimate_kv_cache_memory(
    num_layers=32, num_heads=32, head_dim=128,
    max_seq_len=4096, dtype_size=2
)
for k, v in kv_info.items():
    print(f"  {k}: {v:.4f}" if isinstance(v, float) else f"  {k}: {v}")

print("\n使用 GQA (8 KV heads) 后:")
kv_info_gqa = estimate_kv_cache_memory(
    num_layers=32, num_heads=8, head_dim=128,
    max_seq_len=4096, dtype_size=2
)
for k, v in kv_info_gqa.items():
    print(f"  {k}: {v:.4f}" if isinstance(v, float) else f"  {k}: {v}")

In [ ]:
# ============================================================
# 2.3 内存泄漏检测
# ============================================================

def detect_memory_leak(iterations: int = 20):
    """
    通过循环运行推理并记录每次迭代后的内存占用，
    检测是否存在内存泄漏（内存持续增长而不释放）。
    """
    print("内存泄漏检测循环（使用虚拟模型）...\n")

    model = nn.Sequential(
        nn.Linear(512, 256),
        nn.ReLU(),
        nn.Linear(256, 128),
        nn.ReLU(),
        nn.Linear(128, 64),
    )

    tracker = MemoryTracker()
    tracker.snapshot("循环开始前")

    memory_history = []
    for i in range(iterations):
        x = torch.randn(32, 512)
        y = model(x)
        loss = y.sum()
        loss.backward()

        tracker.snapshot(f"迭代 {i+1}")
        memory_history.append(tracker.snapshots[-1].allocated_gb)

        if i % 5 == 0:
            gc.collect()

    del model, x, y, loss
    gc.collect()
    tracker.snapshot("清理后")

    print(tracker.report())

    # 分析趋势
    if len(memory_history) >= 3:
        start_avg = sum(memory_history[:3]) / 3
        end_avg = sum(memory_history[-3:]) / 3
        growth = end_avg - start_avg
        print(f"\n内存增长趋势分析:")
        print(f"  前3次平均: {start_avg:.6f} GB")
        print(f"  后3次平均: {end_avg:.6f} GB")
        print(f"  增长量:    {growth:.6f} GB")
        if growth > 0.001:
            print(f"  ⚠️  可能存在内存泄漏!")
        else:
            print(f"  ✅ 内存使用稳定，未检测到泄漏")

    return memory_history


leak_result = detect_memory_leak(iterations=15)

In [ ]:
# ============================================================
# 2.4 不同精度下模型大小计算
# ============================================================

def calculate_model_size(num_params: int) -> Dict[str, float]:
    """
    计算不同精度下的模型大小。

    常用精度及字节数:
        FP32 = 4 bytes/param
        FP16/BF16 = 2 bytes/param
        INT8 = 1 byte/param
        INT4 = 0.5 bytes/param
    """
    return {
        "参数数量": num_params,
        "FP32 (GB)": num_params * 4 / (1024 ** 3),
        "FP16 (GB)": num_params * 2 / (1024 ** 3),
        "INT8 (GB)": num_params * 1 / (1024 ** 3),
        "INT4 (GB)": num_params * 0.5 / (1024 ** 3),
    }


# 常见模型参数量
models_config = {
    "TinyLlama-1.1B": 1.1e9,
    "Phi-2 (2.7B)": 2.7e9,
    "Qwen2-7B": 7e9,
    "LLaMA-3-8B": 8e9,
}

print("=" * 80)
print(f"{'模型':<20} {'FP32(GB)':<12} {'FP16(GB)':<12} {'INT8(GB)':<12} {'INT4(GB)':<12}")
print("-" * 80)
for model_name, params in models_config.items():
    sizes = calculate_model_size(int(params))
    print(
        f"{model_name:<20} {sizes['FP32 (GB)']:<12.2f} "
        f"{sizes['FP16 (GB)']:<12.2f} {sizes['INT8 (GB)']:<12.2f} "
        f"{sizes['INT4 (GB)']:<12.2f}"
    )
print("=" * 80)
print("\n💡 提示: 端侧设备通常只有 4-16GB 内存，"
      "INT4 量化是将 7B+ 模型部署到手机的关键技术。")

---

## 3. 精度异常排查 (Precision Anomalies)

当量化后的模型输出与原始模型差异过大时，需要逐层定位问题。本节演示：
- 逐层余弦相似度比较
- 异常通道检测
- 识别量化误差大的层
- 混合精度策略决策

In [ ]:
# ============================================================
# 3.1 构建虚拟的 FP 模型和 "量化" 模型
# ============================================================

class DummyTransformerBlock(nn.Module):
    """虚拟 Transformer 块，用于演示精度排查"""

    def __init__(self, hidden_dim: int = 256, ffn_dim: int = 1024,
                 name: str = ""):
        super().__init__()
        self.name = name
        self.attn_q = nn.Linear(hidden_dim, hidden_dim)
        self.attn_k = nn.Linear(hidden_dim, hidden_dim)
        self.attn_v = nn.Linear(hidden_dim, hidden_dim)
        self.attn_o = nn.Linear(hidden_dim, hidden_dim)
        self.ffn_up = nn.Linear(hidden_dim, ffn_dim)
        self.ffn_down = nn.Linear(ffn_dim, hidden_dim)

    def forward(self, x):
        q = self.attn_q(x)
        k = self.attn_k(x)
        v = self.attn_v(x)
        attn_out = self.attn_o(q)
        ffn_out = self.ffn_down(F.relu(self.ffn_up(attn_out)))
        return ffn_out + x


class DummyLLM(nn.Module):
    """虚拟 LLM 模型，包含多个 Transformer Block"""

    def __init__(self, num_layers: int = 4, hidden_dim: int = 256):
        super().__init__()
        self.embed = nn.Linear(hidden_dim, hidden_dim)
        self.layers = nn.ModuleList([
            DummyTransformerBlock(hidden_dim, name=f"layer_{i}")
            for i in range(num_layers)
        ])
        self.lm_head = nn.Linear(hidden_dim, hidden_dim)

    def forward(self, x):
        x = self.embed(x)
        for layer in self.layers:
            x = layer(x)
        return self.lm_head(x)


# 创建 FP32 参考模型
fp_model = DummyLLM(num_layers=4, hidden_dim=256)
fp_model.eval()

# 模拟量化模型：在权重上添加噪声来模拟量化误差
def simulate_quantization(model: nn.Module, noise_scale: float = 0.01) -> nn.Module:
    """
    通过在权重上添加噪声来模拟量化引入的误差。
    某些层添加更大噪声来模拟高量化误差层。
    """
    import copy
    q_model = copy.deepcopy(model)
    for name, param in q_model.named_parameters():
        if 'weight' in name:
            # 模拟不同层的量化误差
            if 'ffn_up' in name:
                noise = torch.randn_like(param) * noise_scale * 5
            elif 'attn_v' in name:
                noise = torch.randn_like(param) * noise_scale * 3
            else:
                noise = torch.randn_like(param) * noise_scale
            param.data.add_(noise)
    return q_model


q_model = simulate_quantization(fp_model, noise_scale=0.02)
q_model.eval()

print(f"FP 模型参数量: {sum(p.numel() for p in fp_model.parameters()):,}")
print(f"量化模型参数量: {sum(p.numel() for p in q_model.parameters()):,}")
print("模型已就绪。")

In [ ]:
# ============================================================
# 3.2 逐层精度比较
# ============================================================

def compare_layer_outputs(
    fp_model: nn.Module,
    q_model: nn.Module,
    input_tensor: torch.Tensor
) -> List[PrecisionReport]:
    """
    逐层比较 FP 模型和量化模型的输出，
    使用余弦相似度、MSE 和最大绝对误差作为指标。
    """
    checker = PrecisionChecker(fp_model, q_model)

    # 注册 hooks 收集中间输出
    fp_outputs = {}
    q_outputs = {}

    def make_hook(output_dict, name):
        def hook(module, input, output):
            output_dict[name] = output.detach().clone()
        return hook

    fp_handles = []
    q_handles = []
    for name, module in fp_model.named_modules():
        if name == "":
            continue
        fp_handles.append(module.register_forward_hook(make_hook(fp_outputs, name)))
    for name, module in q_model.named_modules():
        if name == "":
            continue
        q_handles.append(module.register_forward_hook(make_hook(q_outputs, name)))

    # 前向传播
    with torch.no_grad():
        fp_model(input_tensor)
        q_model(input_tensor)

    # 移除 hooks
    for h in fp_handles + q_handles:
        h.remove()

    # 比较每层输出
    reports = []
    for name in fp_outputs:
        if name in q_outputs:
            fp_out = fp_outputs[name]
            q_out = q_outputs[name]
            report = checker.compare_layer(name, fp_out, q_out)
            reports.append(report)

    return reports


# 执行比较
test_input = torch.randn(1, 32, 256)
reports = compare_layer_outputs(fp_model, q_model, test_input)

print("逐层精度比较结果：")
print("-" * 80)
print(f"{'层名称':<30} {'余弦相似度':<14} {'MSE':<14} {'异常比例':<10} {'建议'}")
print("-" * 80)
for r in sorted(reports, key=lambda x: x.cosine_sim):
    print(f"{r.layer_name:<30} {r.cosine_sim:<14.6f} {r.mse:<14.6f} "
          f"{r.outlier_ratio:<10.4f} {r.recommendation}")

In [ ]:
# ============================================================
# 3.3 异常通道检测详解
# ============================================================

def analyze_activation_outliers(tensor: torch.Tensor,
                                layer_name: str = ""):
    """
    深入分析激活值中的异常通道。
    异常通道在量化时会导致较大的量化误差。
    """
    print(f"\n激活值异常通道分析: {layer_name}")
    print("-" * 50)
    print(f"  张量形状: {tensor.shape}")
    print(f"  数据类型: {tensor.dtype}")
    print(f"  值范围: [{tensor.min().item():.4f}, {tensor.max().item():.4f}]")
    print(f"  均值: {tensor.mean().item():.4f}")
    print(f"  标准差: {tensor.std().item():.4f}")

    # 计算每个通道的范围
    if tensor.dim() >= 2:
        # 沿最后一个维度计算每个通道的最大绝对值
        channel_max = tensor.float().abs().max(dim=-1).values
        if tensor.dim() > 2:
            channel_max = channel_max.max(dim=-1).values

        threshold_3sigma = channel_max.mean() + 3 * channel_max.std()
        threshold_5sigma = channel_max.mean() + 5 * channel_max.std()

        outliers_3sigma = (channel_max > threshold_3sigma).sum().item()
        outliers_5sigma = (channel_max > threshold_5sigma).sum().item()

        print(f"  通道数: {channel_max.numel()}")
        print(f"  3σ 阈值: {threshold_3sigma:.4f}")
        print(f"  3σ 以上异常通道: {outliers_3sigma} "
              f"({100*outliers_3sigma/channel_max.numel():.2f}%)")
        print(f"  5σ 以上异常通道: {outliers_5sigma} "
              f"({100*outliers_5sigma/channel_max.numel():.2f}%)")

        if outliers_3sigma > 0:
            top_values, top_indices = torch.topk(
                channel_max.flatten(), min(5, channel_max.numel())
            )
            print(f"  前5个最大通道值:")
            for idx, val in zip(top_indices, top_values):
                print(f"    通道 {idx.item()}: {val.item():.4f}")


# 创建含有异常值的激活张量（模拟真实 LLM 中的异常通道现象）
normal_activation = torch.randn(32, 256) * 0.5
analyze_activation_outliers(normal_activation, "正常激活")

# 在少数通道注入异常值
abnormal_activation = normal_activation.clone()
abnormal_activation[0, 10] = 15.0
abnormal_activation[5, 50] = -12.0
abnormal_activation[10, 100] = 20.0
abnormal_activation[20, 200] = 18.0
analyze_activation_outliers(abnormal_activation, "含异常通道的激活")

In [ ]:
# ============================================================
# 3.4 混合精度策略：识别应保留 FP16 的层
# ============================================================

def recommend_mixed_precision_strategy(
    reports: List[PrecisionReport],
    cos_sim_threshold: float = 0.995
) -> Dict[str, List[str]]:
    """
    基于精度报告，推荐混合精度策略。
    余弦相似度低于阈值的层建议保留 FP16。
    """
    strategy = {
        "keep_fp16": [],
        "safe_int8": [],
        "borderline": [],
    }

    for r in reports:
        if r.cosine_sim < 0.99:
            strategy["keep_fp16"].append(r.layer_name)
        elif r.cosine_sim < cos_sim_threshold:
            strategy["borderline"].append(r.layer_name)
        else:
            strategy["safe_int8"].append(r.layer_name)

    return strategy


strategy = recommend_mixed_precision_strategy(reports)

print("=" * 60)
print("混合精度策略推荐")
print("=" * 60)

print(f"\n🔴 建议保留 FP16 ({len(strategy['keep_fp16'])} 层):")
for name in strategy['keep_fp16']:
    r = next(x for x in reports if x.layer_name == name)
    print(f"  - {name} (余弦相似度: {r.cosine_sim:.6f})")

print(f"\n🟡 边界层 ({len(strategy['borderline'])} 层):")
for name in strategy['borderline']:
    r = next(x for x in reports if x.layer_name == name)
    print(f"  - {name} (余弦相似度: {r.cosine_sim:.6f})")

print(f"\n🟢 可安全量化 ({len(strategy['safe_int8'])} 层):")
for name in strategy['safe_int8']:
    r = next(x for x in reports if x.layer_name == name)
    print(f"  - {name} (余弦相似度: {r.cosine_sim:.6f})")

print(f"\n💡 混合精度策略可减少精度损失，同时保持大部分层的量化收益。")
print(f"   FP16 保留比例: {len(strategy['keep_fp16'])/len(reports)*100:.1f}%")

---

## 4. 性能瓶颈分析 (Performance Bottlenecks)

部署模型后推理速度不达标？本节提供工具：
- 逐层计时分析
- MAC 利用率计算
- 瓶颈层 Top-5 报告
- Batch Size 与吞吐量关系

In [ ]:
# ============================================================
# 4.1 逐层性能分析
# ============================================================

# 构建更深一点的虚拟模型用于性能分析
perf_model = DummyLLM(num_layers=6, hidden_dim=256)
perf_model.eval()

profiler = LayerProfiler()
profiler.attach(perf_model)

# 预热
with torch.no_grad():
    for _ in range(3):
        _ = perf_model(torch.randn(1, 32, 256))

# 正式测量
profiler.profiles.clear()
with torch.no_grad():
    for _ in range(10):
        _ = perf_model(torch.randn(1, 32, 256))

profiler.detach()
profiler.profiles = {
    k: v for k, v in profiler.profiles.items()
}

print(profiler.report(top_k=10))

In [ ]:
# ============================================================
# 4.2 MAC 利用率计算
# ============================================================

def estimate_macs_linear(in_features: int, out_features: int,
                         batch_size: int = 1, seq_len: int = 1) -> int:
    """估算 Linear 层的 MACs (Multiply-Accumulate Operations)"""
    return batch_size * seq_len * in_features * out_features


def estimate_macs_attention(hidden_dim: int, seq_len: int,
                            batch_size: int = 1) -> int:
    """估算 Self-Attention 的 MACs"""
    # QKV 投影: 3 * hidden * hidden
    qkv_macs = 3 * batch_size * seq_len * hidden_dim * hidden_dim
    # Attention 矩阵: seq * seq * hidden
    attn_macs = batch_size * seq_len * seq_len * hidden_dim
    # Output 投影: hidden * hidden
    out_macs = batch_size * seq_len * hidden_dim * hidden_dim
    return qkv_macs + attn_macs + out_macs


def calculate_mac_utilization(
    total_macs: int,
    total_time_ms: float,
    theoretical_tops: float = 1.0
) -> Dict[str, float]:
    """
    计算 MAC 利用率。

    参数:
        total_macs: 总 MAC 操作数
        total_time_ms: 总执行时间（毫秒）
        theoretical_tops: 理论峰值算力（TOPS，默认 1.0 用于 CPU）
    """
    achieved_tops = (total_macs / (total_time_ms / 1000)) / 1e12
    utilization = (achieved_tops / theoretical_tops) * 100 if theoretical_tops > 0 else 0

    return {
        "总 MACs (G)": total_macs / 1e9,
        "总耗时 (ms)": total_time_ms,
        "实际 TOPS": achieved_tops,
        "理论 TOPS": theoretical_tops,
        "MAC 利用率 (%)": utilization,
    }


# 估算虚拟模型的 MACs
hidden_dim = 256
seq_len = 32
num_layers = 6

# 每层: 4个 Linear(Q,K,V,O) + 2个 FFN Linear
# Self-Attention MACs + FFN MACs
per_layer_attn = estimate_macs_attention(hidden_dim, seq_len)
per_layer_ffn = (
    estimate_macs_linear(hidden_dim, 4 * hidden_dim, 1, seq_len) +
    estimate_macs_linear(4 * hidden_dim, hidden_dim, 1, seq_len)
)
total_macs = num_layers * (per_layer_attn + per_layer_ffn)

total_time = sum(p.time_ms for p in profiler.profiles.values())
utilization = calculate_mac_utilization(total_macs, total_time, theoretical_tops=1.0)

print("MAC 利用率分析:")
for k, v in utilization.items():
    print(f"  {k}: {v:.4f}" if isinstance(v, float) else f"  {k}: {v}")
print("\n💡 在 CPU 上，MAC 利用率通常很低（<1%），"
      "这是正常的。端侧 NPU 可达 30-80%。")

In [ ]:
# ============================================================
# 4.3 Batch Size 与吞吐量关系
# ============================================================

def benchmark_batch_size_sweep(
    model: nn.Module,
    input_dim: int,
    seq_len: int,
    batch_sizes: List[int],
    warmup: int = 3,
    repeats: int = 10
) -> Dict[int, Dict[str, float]]:
    """
    测试不同 batch size 下的吞吐量。
    寻找最佳 batch size 以平衡延迟和吞吐。
    """
    results = {}

    for bs in batch_sizes:
        inputs = torch.randn(bs, seq_len, input_dim)

        # 预热
        with torch.no_grad():
            for _ in range(warmup):
                _ = model(inputs)

        # 测量
        start = time.time()
        with torch.no_grad():
            for _ in range(repeats):
                _ = model(inputs)
        elapsed = time.time() - start

        avg_time_ms = (elapsed / repeats) * 1000
        throughput = bs / (avg_time_ms / 1000)

        results[bs] = {
            "平均延迟 (ms)": avg_time_ms,
            "吞吐量 (samples/s)": throughput,
        }

    return results


print("Batch Size 与吞吐量关系测试...")
batch_sizes = [1, 2, 4, 8, 16, 32]
results = benchmark_batch_size_sweep(
    perf_model, input_dim=256, seq_len=32,
    batch_sizes=batch_sizes, warmup=2, repeats=5
)

print(f"\n{'Batch Size':<12} {'平均延迟(ms)':<16} {'吞吐量(samples/s)':<20}")
print("-" * 50)
for bs in batch_sizes:
    r = results[bs]
    print(f"{bs:<12} {r['平均延迟 (ms)']:<16.2f} {r['吞吐量 (samples/s)']:<20.2f}")

---

## 5. 框架与编译错误排查 (Framework Errors)

模型导出到 ONNX 或使用 torch.export 时经常会遇到各种错误。本节演示：
- ONNX 导出常见问题
- torch.export vs torch.onnx.export
- 算子分解
- CPU 回退成本计算

In [ ]:
# ============================================================
# 5.1 动态形状问题与 ONNX 导出
# ============================================================

print("ONNX 导出诊断工具")
print("=" * 60)


class ONNXExportDiagnostic:
    """ONNX 导出诊断助手"""

    @staticmethod
    def check_dynamic_shapes(model: nn.Module,
                             sample_input: torch.Tensor):
        """检查模型是否依赖动态形状操作"""
        issues = []

        for name, module in model.named_modules():
            module_type = type(module).__name__

            # 检查已知的有问题的模式
            if module_type == "Conditional" or "if" in name.lower():
                issues.append(
                    f"⚠️  {name}: {module_type} — "
                    f"条件分支可能导致 ONNX 导出困难"
                )

            # 检查 view/reshape 中的动态维度
            if hasattr(module, 'extra_repr'):
                extra = module.extra_repr()
                if '-1' in extra:
                    issues.append(
                        f"⚠️  {name}: {module_type} — "
                        f"包含动态维度 (-1)，建议指定具体值"
                    )

        return issues

    @staticmethod
    def common_onnx_errors() -> Dict[str, str]:
        """ONNX 导出常见错误及解决方案"""
        return {
            "RuntimeError: shape mismatch":
                "检查输入张量的形状是否与模型期望一致。使用 torch.onnx.export 时指定 dynamic_axes。",
            "RuntimeError: Unsupported operator":
                "使用 torch.onnx.export 的 operator_export_type 参数或手动替换不支持的算子。",
            "TracerWarning: Converting a tensor to a Python boolean":
                "避免在 forward 中使用 Python 条件判断。将条件逻辑转为 torch.where。",
            "RuntimeError: Cannot export deterministic convolution":
                "设置 torch.backends.cudnn.deterministic = True。",
            "Exporting the operator 'xxx' to ONNX":
                "检查 ONNX opset 版本。使用较新的 opset_version (≥14)。",
        }


# 检查虚拟模型
diagnostic = ONNXExportDiagnostic()
sample_input = torch.randn(1, 32, 256)
issues = diagnostic.check_dynamic_shapes(fp_model, sample_input)

print("模型检查结果:")
if issues:
    for issue in issues:
        print(f"  {issue}")
else:
    print("  ✅ 未检测到明显的动态形状问题。")

print("\nONNX 导出常见错误速查:")
for error, solution in diagnostic.common_onnx_errors().items():
    print(f"  错误: {error}")
    print(f"  解决: {solution}")
    print()

In [ ]:
# ============================================================
# 5.2 torch.export vs torch.onnx.export
# ============================================================

print("torch.export vs torch.onnx.export 对比")
print("=" * 60)

# 构建一个简单的导出友好模型
class ExportFriendlyModel(nn.Module):
    """
    导出友好的模型：
    - 使用 torch.where 替代 if/else
    - 避免动态 shape 操作
    - 所有尺寸在 forward 中可追踪
    """
    def __init__(self):
        super().__init__()
        self.linear1 = nn.Linear(64, 128)
        self.linear2 = nn.Linear(128, 64)

    def forward(self, x):
        x = self.linear1(x)
        x = F.relu(x)
        x = self.linear2(x)
        return x


export_model = ExportFriendlyModel()
export_model.eval()
example_input = torch.randn(4, 64)

# 尝试 torch.onnx.export
print("\n1. 尝试 torch.onnx.export:")
try:
    import io
    buffer = io.BytesIO()
    torch.onnx.export(
        export_model,
        example_input,
        buffer,
        input_names=['input'],
        output_names=['output'],
        dynamic_axes={'input': {0: 'batch'}, 'output': {0: 'batch'}},
        opset_version=14,
    )
    print("  ✅ torch.onnx.export 成功！")
    print(f"  ONNX 模型大小: {len(buffer.getvalue()):,} bytes")
except Exception as e:
    print(f"  ❌ torch.onnx.export 失败: {e}")

# 尝试 torch.export (PyTorch 2.0+)
print("\n2. 尝试 torch.export.export:")
try:
    exported_program = torch.export.export(export_model, (example_input,))
    print("  ✅ torch.export.export 成功！")
    print(f"  导出图的节点数: {len(list(exported_program.graph.nodes))}")
except Exception as e:
    print(f"  ⚠️  torch.export.export 不可用或失败: {e}")

print("\n方法选择指南:")
print("  torch.onnx.export: 适用于通用 ONNX 生态（ORT、QNN、SNPE 等）")
print("  torch.export: PyTorch 2.0+ 推荐，提供更严格的图 IR")
print("  torch.compile: 适用于在 PyTorch 原生环境加速推理")

In [ ]:
# ============================================================
# 5.3 算子分解示例
# ============================================================

print("算子分解 (Operator Decomposition)")
print("=" * 60)


def decompose_gelu(x: torch.Tensor) -> torch.Tensor:
    """
    将 GELU 激活函数分解为基础算子。
    某些推理引擎不支持 GELU，需要手动分解为 tanh/erf 组合。

    GELU(x) ≈ 0.5 * x * (1 + tanh(sqrt(2/π) * (x + 0.044715 * x^3)))
    """
    return (
        0.5 * x * (
            1.0 + torch.tanh(
                np.sqrt(2.0 / np.pi) * (x + 0.044715 * torch.pow(x, 3))
            )
        )
    )


x = torch.randn(100, device='cpu')
gelu_ref = F.gelu(x)
gelu_decomposed = decompose_gelu(x)
diff = (gelu_ref - gelu_decomposed).abs().max().item()

print(f"  GELU 原始输出 vs 分解输出最大误差: {diff:.8f}")
print(f"  ✅ 误差可忽略，分解方案可行")


# 常见需要分解的算子
unsupported_ops = {
    "torch.nn.functional.gelu": "→ gelu_tanh_approx(x)",
    "torch.nn.functional.layer_norm": "→ (x - mean) / sqrt(var + eps) * weight + bias",
    "torch.nn.functional.scaled_dot_product_attention":
        "→ 手动拆分 Q*K^T / sqrt(d) → softmax → *V",
    "torch.nn.functional.rms_norm":
        "→ x / sqrt(mean(x^2) + eps) * weight",
    ":= (in-place assignment)": "→ 使用标准赋值并手动管理",
}

print("\n常见需要分解的算子速查表:")
for op, replacement in unsupported_ops.items():
    print(f"  {op}")
    print(f"    {replacement}")

In [ ]:
# ============================================================
# 5.4 CPU 回退成本计算
# ============================================================

def calculate_cpu_fallback_cost(
    num_fallback_ops: int,
    total_ops: int,
    npu_time_per_op_ms: float = 0.01,
    cpu_time_per_op_ms: float = 1.0,
    data_transfer_ms: float = 5.0
) -> Dict[str, float]:
    """
    计算因部分算子不支持 NPU 而回退到 CPU 的性能损失。

    参数:
        num_fallback_ops: 需要回退的算子数量
        total_ops: 总算子数量
        npu_time_per_op_ms: NPU 上每个算子的平均执行时间
        cpu_time_per_op_ms: CPU 上每个算子的平均执行时间
        data_transfer_ms: 每次 CPU↔NPU 数据搬运开销
    """
    ops_on_npu = total_ops - num_fallback_ops
    npu_only_time = total_ops * npu_time_per_op_ms
    mixed_time = (
        ops_on_npu * npu_time_per_op_ms +
        num_fallback_ops * cpu_time_per_op_ms +
        num_fallback_ops * 2 * data_transfer_ms  # 搬运到CPU再搬回NPU
    )

    slowdown = mixed_time / npu_only_time if npu_only_time > 0 else 1.0

    return {
        "回退算子数": num_fallback_ops,
        "回退比例": num_fallback_ops / total_ops * 100 if total_ops > 0 else 0,
        "纯 NPU 耗时 (ms)": npu_only_time,
        "混合执行耗时 (ms)": mixed_time,
        "性能下降倍数": slowdown,
        "数据搬运开销 (ms)": num_fallback_ops * 2 * data_transfer_ms,
    }


# 测试场景
print("CPU 回退成本分析")
print("=" * 60)

scenarios = [
    ("所有算子均支持 (理想)", 0, 100),
    ("5% 算子回退", 5, 100),
    ("20% 算子回退", 20, 100),
    ("50% 算子回退 (严重)", 50, 100),
]

for name, fallback, total in scenarios:
    result = calculate_cpu_fallback_cost(fallback, total)
    print(f"\n📊 {name}:")
    print(f"   回退比例: {result['回退比例']:.1f}%")
    print(f"   混合执行耗时: {result['混合执行耗时 (ms)']:.2f} ms")
    print(f"   性能下降: {result['性能下降倍数']:.2f}x")
    print(f"   数据搬运: {result['数据搬运开销 (ms)']:.2f} ms")

print("\n💡 结论: 即使只有 5% 的算子回退 CPU，"
      "由于数据搬运开销，性能可能下降数倍。")
print("   因此算子覆盖率是端侧部署的关键指标。")

---

## 6. 端到端诊断工作流 (End-to-End Diagnostic Workflow)

将前面的所有诊断工具整合为一个统一的诊断函数，
一键生成综合诊断报告和常见问题的决策树。

In [ ]:
# ============================================================
# 6.1 综合诊断函数
# ============================================================

@dataclass
class DiagnosticReport:
    """综合诊断报告"""
    model_name: str = ""
    total_params: int = 0
    memory_ok: bool = True
    precision_ok: bool = True
    performance_ok: bool = True
    top_bottlenecks: List[str] = field(default_factory=list)
    high_error_layers: List[str] = field(default_factory=list)
    recommendations: List[str] = field(default_factory=list)
    warnings: List[str] = field(default_factory=list)


def run_full_diagnostic(
    model: nn.Module,
    model_name: str = "MyModel",
    sample_input: Optional[torch.Tensor] = None,
    available_memory_gb: float = 8.0,
) -> DiagnosticReport:
    """
    端到端诊断工作流：
    1. 内存评估
    2. 性能分析
    3. 精度检查（如提供量化模型）
    4. 生成综合报告
    """
    print(f"\n{'='*60}")
    print(f"正在运行端到端诊断: {model_name}")
    print(f"{'='*60}\n")

    report = DiagnosticReport(model_name=model_name)

    # Step 1: 内存评估
    print("[1/4] 内存评估...")
    total_params = sum(p.numel() for p in model.parameters())
    report.total_params = total_params
    size_fp16 = total_params * 2 / (1024 ** 3)
    size_int4 = total_params * 0.5 / (1024 ** 3)

    print(f"  参数总量: {total_params:,}")
    print(f"  FP16 模型大小: {size_fp16:.4f} GB")
    print(f"  INT4 模型大小: {size_int4:.4f} GB")

    if size_fp16 > available_memory_gb * 0.8:
        report.warnings.append(
            f"FP16 模型 ({size_fp16:.2f}GB) 接近内存上限"
            f" ({available_memory_gb}GB)，建议使用 INT8/INT4 量化"
        )
        report.memory_ok = False
    else:
        print(f"  ✅ 内存评估通过 (FP16: {size_fp16:.2f} GB ≤ "
              f"{available_memory_gb * 0.8:.2f} GB)")

    # Step 2: 性能分析
    print("\n[2/4] 性能分析...")
    if sample_input is None:
        sample_input = torch.randn(1, 32, 256)

    profiler = LayerProfiler()
    profiler.attach(model)

    with torch.no_grad():
        for _ in range(5):
            _ = model(sample_input)

    profiler.detach()
    bottlenecks = profiler.get_top_bottlenecks(k=5)
    report.top_bottlenecks = [
        f"{b.name} ({b.time_ms:.2f}ms)" for b in bottlenecks
    ]

    print(f"  最耗时的层:")
    for b in bottlenecks:
        print(f"    - {b.name}: {b.time_ms:.4f} ms")

    # Step 3: 结构检查
    print("\n[3/4] 模型结构检查...")
    module_types = defaultdict(int)
    for _, module in model.named_modules():
        module_types[type(module).__name__] += 1

    print(f"  模块类型分布:")
    for mtype, count in module_types.items():
        if count > 0:
            print(f"    - {mtype}: {count}")

    # 检查可能不兼容的算子
    incompatible_patterns = []
    for name, module in model.named_modules():
        if 'Dropout' in type(module).__name__:
            incompatible_patterns.append(
                f"{name}: Dropout 在推理时应设为 eval 模式"
            )

    if incompatible_patterns:
        report.warnings.extend(incompatible_patterns[:3])

    # Step 4: 生成推荐
    print("\n[4/4] 生成推荐...")
    if size_fp16 > available_memory_gb * 0.5:
        report.recommendations.append("使用 INT8 或 INT4 量化减少内存占用")
    if bottlenecks:
        top_bottleneck = bottlenecks[0]
        report.recommendations.append(
            f"优先优化 {top_bottleneck.name} (占比最大)"
        )
    report.recommendations.append("导出前确保模型处于 eval() 模式")
    report.recommendations.append("使用 torch.inference_mode() 替代 torch.no_grad()")

    # 打印最终报告
    print(f"\n{'='*60}")
    print(f"诊断报告: {model_name}")
    print(f"{'='*60}")
    print(f"  参数量: {report.total_params:,}")
    print(f"  内存状态: {'⚠️ 需优化' if not report.memory_ok else '✅ 正常'}")
    print(f"  性能瓶颈:")
    for b in report.top_bottlenecks:
        print(f"    - {b}")
    print(f"  建议:")
    for i, rec in enumerate(report.recommendations, 1):
        print(f"    {i}. {rec}")
    if report.warnings:
        print(f"  警告:")
        for w in report.warnings:
            print(f"    ⚠️  {w}")
    print(f"{'='*60}")

    return report


# 运行端到端诊断
full_report = run_full_diagnostic(
    fp_model,
    model_name="DummyLLM-256",
    sample_input=torch.randn(1, 32, 256),
    available_memory_gb=8.0,
)

In [ ]:
# ============================================================
# 6.2 常见问题决策树
# ============================================================

def decision_tree_guide(issue_type: str = "all"):
    """
    端侧部署常见问题决策树。

    用法: 根据遇到的症状，选择对应的问题类型，
          打印排查步骤和建议。
    """
    tree = {
        "OOM": {
            "症状": "部署时出现 CUDA out of memory 或设备内存不足",
            "排查步骤": [
                "1. 计算模型大小: params * dtype_bytes / (1024^3)",
                "2. 估算 KV Cache: 2 * layers * heads * head_dim * seq_len * dtype_bytes",
                "3. 检查运行时开销: 模型大小的 20-30%",
                "4. 总需求 = 模型大小 + KV Cache + 运行时开销",
            ],
            "解决方案": [
                "→ 使用更小的模型或 INT4 量化",
                "→ 减少 max_seq_len 以缩小 KV Cache",
                "→ 启用 GQA/MQA 减少 KV 头数",
                "→ 使用 FlashAttention 减少中间激活内存",
            ]
        },
        "精度异常": {
            "症状": "量化后模型输出与原始模型差异过大",
            "排查步骤": [
                "1. 逐层比较 FP32 和量化输出的余弦相似度",
                "2. 检测激活值中的异常通道（>5σ）",
                "3. 检查量化参数 (scale, zero_point) 是否合理",
                "4. 验证校准数据集是否与推理分布一致",
            ],
            "解决方案": [
                "→ 对敏感层使用混合精度（保留 FP16）",
                "→ 使用 SmoothQuant 迁移量化难度",
                "→ 使用 per-channel 量化替代 per-tensor",
                "→ 增加校准样本数量和多样性",
            ]
        },
        "性能不达标": {
            "症状": "推理速度远低于预期",
            "排查步骤": [
                "1. 使用 LayerProfiler 找到最耗时的层",
                "2. 检查 MAC 利用率是否过低",
                "3. 核查是否有 CPU 回退算子",
                "4. 确认推理框架是否启用图优化",
            ],
            "解决方案": [
                "→ 使用 torch.compile / ONNX Runtime 图优化",
                "→ 替换/分解不支持的算子避免 CPU 回退",
                "→ 调整 batch size 优化吞吐",
                "→ 启用 FP16/INT8 计算（需硬件支持）",
            ]
        },
        "导出失败": {
            "症状": "ONNX 导出或 torch.export 报错",
            "排查步骤": [
                "1. 检查是否使用了动态控制流 (if/for)",
                "2. 确认不支持的操作是否可分解",
                "3. 尝试更新的 opset 版本",
                "4. 使用 torch.export 的严格模式定位问题",
            ],
            "解决方案": [
                "→ 将条件逻辑改写为 torch.where",
                "→ 手动分解不支持的算子",
                "→ 升级 ONNX opset 到 17+",
                "→ 对无法分解的算子标记为 CPU 回退",
            ]
        },
    }

    if issue_type == "all":
        for category, info in tree.items():
            print(f"\n{'='*60}")
            print(f"🐛 问题类型: {category}")
            print(f"{'='*60}")
            print(f"  症状: {info['症状']}")
            print(f"\n  排查步骤:")
            for step in info['排查步骤']:
                print(f"    {step}")
            print(f"\n  解决方案:")
            for sol in info['解决方案']:
                print(f"    {sol}")
    elif issue_type in tree:
        info = tree[issue_type]
        print(f"\n{'='*60}")
        print(f"🐛 问题类型: {issue_type}")
        print(f"{'='*60}")
        print(f"  症状: {info['症状']}")
        print(f"\n  排查步骤:")
        for step in info['排查步骤']:
            print(f"    {step}")
        print(f"\n  解决方案:")
        for sol in info['解决方案']:
            print(f"    {sol}")
    else:
        print(f"未知问题类型: {issue_type}")
        print(f"可用类型: {list(tree.keys())}")


# 打印完整决策树
decision_tree_guide("all")

---

## 7. 调试工具参考 (Debugging Tools Reference)

本节汇总端侧部署中常用的调试和分析工具，
包括 PyTorch Profiler、QNN Profiler、ONNX Runtime Profiling 等。

In [ ]:
# ============================================================
# 7.1 调试工具总览表
# ============================================================

tools_reference = [
    {
        "工具": "PyTorch Profiler",
        "适用场景": "PyTorch 模型的 CPU/GPU 性能分析",
        "关键功能": "算子级时间线、内存分析、FLOPs 统计",
        "输出格式": "Chrome Trace JSON, TensorBoard",
        "使用复杂度": "低",
    },
    {
        "工具": "QNN Profiler",
        "适用场景": "Qualcomm 端侧 NPU/DSP 性能分析",
        "关键功能": "算子映射、NPU 利用率、内存带宽分析",
        "输出格式": "qnn-profiler CSV, HTML Report",
        "使用复杂度": "中",
    },
    {
        "工具": "msprof (MindSpore)",
        "适用场景": "Ascend NPU 端侧性能分析",
        "关键功能": "算子耗时、通信开销、内存使用",
        "输出格式": "Timeline JSON, Summary CSV",
        "使用复杂度": "中",
    },
    {
        "工具": "Instruments (Apple)",
        "适用场景": "iOS/macOS Core ML 模型性能分析",
        "关键功能": "ANE 利用率、内存占用、能耗分析",
        "输出格式": "Instruments Trace",
        "使用复杂度": "低",
    },
    {
        "工具": "ONNX Runtime Profiling",
        "适用场景": "ONNX 模型的跨平台性能分析",
        "关键功能": "节点级耗时、内存峰值、图优化效果",
        "输出格式": "JSON 报告",
        "使用复杂度": "低",
    },
    {
        "工具": "Android GPU Inspector",
        "适用场景": "Android GPU/NNAPI 推理性能分析",
        "关键功能": "GPU 计数器、帧时间线、shader 分析",
        "输出格式": "AGI Trace File",
        "使用复杂度": "中",
    },
    {
        "工具": "TFLite Benchmark Tool",
        "适用场景": "TFLite 模型的端侧基准测试",
        "关键功能": "延迟、内存、功耗测量",
        "输出格式": "终端文本输出",
        "使用复杂度": "低",
    },
]

print("=" * 100)
print("端侧部署常用调试工具总览")
print("=" * 100)
print(f"{'工具':<28} {'适用场景':<35} {'关键功能':<35} {'复杂度'}")
print("-" * 100)
for tool in tools_reference:
    print(
        f"{tool['工具']:<28} {tool['适用场景']:<35} "
        f"{tool['关键功能']:<35} {tool['使用复杂度']}"
    )
print("=" * 100)

In [ ]:
# ============================================================
# 7.2 PyTorch Profiler 使用示例
# ============================================================

print("PyTorch Profiler 基本使用")
print("-" * 50)

print("""
# 基本用法 (在 GPU 环境中):

from torch.profiler import profile, record_function, ProfilerActivity

with profile(
    activities=[ProfilerActivity.CPU, ProfilerActivity.CUDA],
    record_shapes=True,
    profile_memory=True,
    with_stack=True,
) as prof:
    with record_function("model_inference"):
        output = model(input_data)

# 打印关键指标
print(prof.key_averages().table(sort_by="cuda_time_total", row_limit=10))

# 导出 Chrome Trace
prof.export_chrome_trace("trace.json")
""")

print("\n💡 提示: 在 CPU 环境中，将 ProfilerActivity.CUDA 改为仅 ProfilerActivity.CPU。")

In [ ]:
# ============================================================
# 7.3 ONNX Runtime Profiling 示例
# ============================================================

print("ONNX Runtime Profiling 使用示例")
print("-" * 50)

print("""
# ONNX Runtime 会话配置启用 profiling:

import onnxruntime as ort

session_options = ort.SessionOptions()
session_options.enable_profiling = True
session_options.profile_file_prefix = "ort_profile"

session = ort.InferenceSession(
    "model.onnx",
    sess_options=session_options,
    providers=['CPUExecutionProvider']
)

# 运行推理
outputs = session.run(None, {'input': input_array})

# Profiling 结果将写入 ort_profile_YYYYMMDDHHMMSS.json
# 可以用 Chrome tracing (chrome://tracing) 打开查看
""")

print("\n💡 ONNX Runtime profiling 输出包含每个节点的精确耗时，")
print("   有助于识别哪些算子是性能瓶颈。")

In [ ]:
# ============================================================
# 7.4 自定义调试工具：快速诊断脚本
# ============================================================

print("自定义快速诊断脚本")
print("-" * 50)

def quick_diagnose(model: nn.Module, input_shape: Tuple[int, ...] = (1, 32, 256)):
    """
    快速诊断：一键获取模型的关键信息。
    适用于部署前的快速检查。
    """
    print("\n⚡ 快速诊断开始...")

    # 基本信息
    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(
        p.numel() for p in model.parameters() if p.requires_grad
    )
    print(f"  参数量: {total_params:,} (可训练: {trainable_params:,})")
    print(f"  FP16 大小: {total_params * 2 / 1e9:.2f} GB")

    # 推理测试
    dummy_input = torch.randn(*input_shape)
    model.eval()

    with torch.inference_mode():
        try:
            start = time.time()
            output = model(dummy_input)
            elapsed = (time.time() - start) * 1000
            print(f"  推理测试: ✅ 成功 ({elapsed:.2f} ms)")
            print(f"  输出形状: {output.shape}")
            if hasattr(output, 'dtype'):
                print(f"  输出类型: {output.dtype}")
        except Exception as e:
            print(f"  推理测试: ❌ 失败 — {e}")

    # 结构概览
    layer_count = len(list(model.named_modules()))
    print(f"  模块数量: {layer_count - 1}")  # -1 排除根模块

    print("  快速诊断完成。\n")


# 测试快速诊断
quick_diagnose(fp_model, input_shape=(1, 32, 256))

---

## 总结

本 Notebook 涵盖了 LLM 端侧部署中的六大故障排查领域：

| 领域 | 关键工具/方法 | 典型问题 | 解决思路 |
|------|-------------|---------|--------|
| 内存 | MemoryTracker, KV Cache 估算 | OOM, 内存泄漏 | 量化、GQA、减少序列长度 |
| 精度 | PrecisionChecker, 余弦相似度 | 量化误差、异常通道 | 混合精度、SmoothQuant |
| 性能 | LayerProfiler, MAC 利用率 | 推理速度慢、算子瓶颈 | 图优化、消除 CPU 回退 |
| 框架 | ONNX 导出诊断、算子分解 | 导出失败、不支持算子 | 算子分解、升级 opset |
| 综合诊断 | run_full_diagnostic | 多因素复合问题 | 按决策树逐项排查 |
| 工具链 | PyTorch Profiler、ORT Profiling | 缺乏可观测性 | 引入 profiling 工具 |

### 关键原则

1. **先测量，再优化** — 使用 profiling 工具定位瓶颈，不要凭感觉优化
2. **分层排查** — 内存→精度→性能→框架，逐步缩小问题范围
3. **保持可观测性** — 在关键路径添加监控点，便于快速定位问题
4. **了解目标硬件** — 不同平台（NPU/GPU/CPU）特性和限制不同
5. **量化是权衡** — 小模型损失精度、大模型消耗内存，根据实际需求选择